[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/085.%20The%20Uncapacitated%20Facility%20Location%20Problem/P85-Tier-9_executed.ipynb)

---


# 85. The Uncapacitated Facility Location Problem
## Tier 9: The Quantum Approximate Optimization Algorithm (QAOA)

### Purpose and scope
This notebook formulates the uncapacitated facility location problem (UFLP) as a quadratic unconstrained binary optimization (QUBO) problem and describes how QAOA uses alternating problem and mixer Hamiltonians, with a classical optimizer tuning their parameters. The demonstration includes a local state-vector simulator and a small-instance classical exhaustive-search benchmark.

The example has 4 facilities and 6 customers. It uses 4 facility-opening binary variables and 24 customer-assignment binary variables, for 28 qubits in the QUBO encoding. Although the QAOA circuit is defined for this instance, a dense state-vector simulation requires memory proportional to \(2^n\), where \(n\) is the number of qubits. The notebook therefore does not launch the 28-qubit simulation: its local simulator deliberately checks a resource limit before allocating the state. This is a computational limitation, not a QAOA result. The classical benchmark and QAOA formulation remain available for study.

The optional AWS Braket configuration is only a placeholder; this notebook does not submit circuits to cloud hardware. It does not establish a quantum speedup or quantum advantage. Such claims require an appropriate implementation, hardware or simulator execution, and a controlled comparison.

### Model assumptions
- Each customer must be assigned to exactly one facility.
- A customer may only be assigned to a facility that is open.
- Facilities have no capacity limits; all costs are represented by fixed opening costs and customer-to-facility transportation costs.
- All binary variables use values in \(\{0,1\}\). The QUBO penalty weight is a modeling parameter intended to discourage constraint violations.

### Method outline
1. Represent facility-opening and customer-assignment decisions as binary variables.
2. Encode the UFLP objective and assignment constraints in a QUBO matrix.
3. Use QAOA layers that alternate problem-Hamiltonian evolution with mixer-Hamiltonian evolution.
4. In principle, use a classical optimizer to tune the circuit angles, estimate the objective from measurements, and decode sampled bitstrings.
5. Interpret sampled solutions only after checking their feasibility and calculating their original UFLP costs.

### What to examine
- How the QUBO penalty terms represent the UFLP constraints.
- Why QAOA uses both a problem Hamiltonian and a mixer Hamiltonian.
- Why a 28-qubit dense state-vector simulation is not executed in this local demonstration.
- How the exact classical benchmark enumerates facility-opening configurations for this small instance.

The numerical results of stochastic QAOA are not specified in advance: they depend on executing the parameter optimization and sampling.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import itertools
import warnings

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize

warnings.filterwarnings("ignore")

# Preserve a reproducible source of randomness for parameter initialization and sampling.
np.random.seed(85)

print("Libraries imported for the QAOA and UFLP demonstration.")

## 1. Problem data and decision variables

Let \(I\) be the set of customers and \(J\) the set of potential facilities. The binary variable \(y_j\) is 1 when facility \(j\) is open, and \(x_{ij}\) is 1 when customer \(i\) is assigned to facility \(j\). The UFLP minimizes fixed opening costs plus transportation costs:

\[
\min_{x,y}\quad \sum_{j\in J} f_j y_j + \sum_{i\in I}\sum_{j\in J} c_{ij}x_{ij}
\]

subject to

\[
\sum_{j\in J}x_{ij}=1 \quad (i\in I), \qquad x_{ij}\leq y_j \quad (i\in I, j\in J), \qquad x_{ij},y_j\in\{0,1\}.
\]

Here, \(f_j\) is the fixed cost of opening facility \(j\), and \(c_{ij}\) is the transportation cost from facility \(j\) to customer \(i\). The first constraint assigns each customer exactly once. The second prevents assignments to closed facilities.

The data class below keeps the dimensions and cost arrays together so that the formulation and solution decoder use the same instance.

In [ ]:
@dataclass
class UFLPInstance:
    """Validated cost data for an uncapacitated facility location instance."""

    n_facilities: int
    n_customers: int
    fixed_costs: List[float]
    transport_costs: np.ndarray

    def __post_init__(self) -> None:
        assert self.n_facilities > 0
        assert self.n_customers > 0
        assert len(self.fixed_costs) == self.n_facilities
        assert self.transport_costs.shape == (self.n_customers, self.n_facilities)


@dataclass
class UFLPSolution:
    """Facility decisions, customer assignments, and original UFLP costs."""

    facilities_open: List[int]
    assignments: List[int]
    total_cost: float
    fixed_cost: float
    transport_cost: float
    feasible: bool


@dataclass
class QAOAParameters:
    """QAOA layer count and classical optimization settings."""

    circuit_depth: int = 2
    shots: int = 1024
    optimizer: str = "COBYLA"
    max_iterations: int = 100
    penalty_weight: float = 1000.0


@dataclass
class QuantumHardwareConfig:
    """Optional hardware settings; no cloud task is submitted in this notebook."""

    use_aws: bool = False
    device_arn: str = ""
    s3_bucket: str = ""
    region: str = "us-east-1"


print("UFLP data structures defined.")

In [ ]:
def create_quantum_example_instance() -> UFLPInstance:
    """Create the four-facility, six-customer teaching instance."""
    fixed_costs = [120, 80, 100, 90]
    transport_costs = np.array(
        [
            [15, 25, 18, 22],
            [20, 12, 28, 15],
            [18, 30, 14, 25],
            [25, 15, 20, 12],
            [22, 18, 25, 20],
            [30, 22, 15, 18],
        ],
        dtype=float,
    )
    return UFLPInstance(
        n_facilities=4,
        n_customers=6,
        fixed_costs=fixed_costs,
        transport_costs=transport_costs,
    )


quantum_instance = create_quantum_example_instance()
print("Example instance:")
print(f"  Facilities: {quantum_instance.n_facilities}")
print(f"  Customers: {quantum_instance.n_customers}")
print(f"  Fixed costs: {quantum_instance.fixed_costs}")
print("  Transportation costs (rows are customers, columns are facilities):")
print(quantum_instance.transport_costs.astype(int))

facility_qubits = quantum_instance.n_facilities
assignment_qubits = quantum_instance.n_customers * quantum_instance.n_facilities
total_qubits = facility_qubits + assignment_qubits
print("QUBO encoding size:")
print(f"  Facility variables: {facility_qubits}")
print(f"  Assignment variables: {assignment_qubits}")
print(f"  Total binary variables / qubits: {total_qubits}")

## 2. QUBO formulation

A QUBO is a binary objective with linear and pairwise terms. A constraint can be incorporated by adding a penalty that is zero when the constraint is satisfied and positive when it is violated. For each customer, the exactly-one assignment penalty is

\[
P\left(\sum_{j\in J}x_{ij}-1\right)^2
= P - P\sum_{j\in J}x_{ij} + 2P\sum_{j<k}x_{ij}x_{ik},
\]

where the equality uses \(x_{ij}^2=x_{ij}\) for binary variables. The constant \(P\) does not affect which assignment minimizes the objective, so it can be omitted from the QUBO matrix. The open-facility penalty is

\[
P x_{ij}(1-y_j)=P x_{ij}-P x_{ij}y_j.
\]

Thus, the QUBO energy is the original cost plus these penalties, up to the omitted constant. A sufficiently strong penalty discourages infeasible assignments; choosing a penalty that is too small can allow violations to be energetically attractive.

The code stores each pairwise coefficient once in the upper-triangular matrix \(Q\), using the convention \(E(z)=\sum_{a\leq b}Q_{ab}z_az_b\). The symmetric matrix representation is deliberately not used: that convention avoids accidentally counting an off-diagonal term twice. The variable-to-qubit map makes the encoding order explicit.

In [ ]:
class QUBOFormulator:
    """Encode a UFLP instance as an upper-triangular QUBO matrix."""

    def __init__(self, instance: UFLPInstance, penalty_weight: float = 1000.0):
        self.instance = instance
        self.penalty_weight = penalty_weight
        self.n_facility_qubits = instance.n_facilities
        self.n_assignment_qubits = instance.n_customers * instance.n_facilities
        self.total_qubits = self.n_facility_qubits + self.n_assignment_qubits

    def get_qubit_indices(self) -> Dict[str, Dict[Tuple[int, ...], int]]:
        """Map facility and assignment indices to their binary-variable positions."""
        mapping = {"facility": {}, "assignment": {}}
        for j in range(self.instance.n_facilities):
            mapping["facility"][(j,)] = j
        for i in range(self.instance.n_customers):
            for j in range(self.instance.n_facilities):
                index = self.n_facility_qubits + i * self.instance.n_facilities + j
                mapping["assignment"][(i, j)] = index
        return mapping

    def formulate_qubo(self) -> np.ndarray:
        """Return Q with energy E(z) = sum_{a <= b} Q[a,b] z[a] z[b]."""
        Q = np.zeros((self.total_qubits, self.total_qubits), dtype=float)
        mapping = self.get_qubit_indices()

        for j in range(self.instance.n_facilities):
            q = mapping["facility"][(j,)]
            Q[q, q] += self.instance.fixed_costs[j]

        for i in range(self.instance.n_customers):
            assignment_qubits = [
                mapping["assignment"][(i, j)]
                for j in range(self.instance.n_facilities)
            ]
            for j, q in enumerate(assignment_qubits):
                Q[q, q] += self.instance.transport_costs[i, j]

            # Expanding P(sum_j x_ij - 1)^2 gives -P on each diagonal,
            # 2P on each pair, and a constant P that does not affect minimizers.
            for q in assignment_qubits:
                Q[q, q] -= self.penalty_weight
            for position, q1 in enumerate(assignment_qubits):
                for q2 in assignment_qubits[position + 1 :]:
                    Q[q1, q2] += 2 * self.penalty_weight

            for j, q in enumerate(assignment_qubits):
                facility_q = mapping["facility"][(j,)]
                # P*x_ij*(1-y_j) penalizes assigning to a closed facility.
                Q[q, q] += self.penalty_weight
                Q[min(q, facility_q), max(q, facility_q)] -= self.penalty_weight

        return Q

    def qubo_energy(self, bitstring: str, qubo_matrix: np.ndarray) -> float:
        """Evaluate the upper-triangular QUBO energy of a measured bitstring."""
        bits = np.fromiter((int(bit) for bit in bitstring), dtype=float)
        if len(bits) != self.total_qubits or not set(bitstring) <= {"0", "1"}:
            raise ValueError("Bitstring length or alphabet does not match the QUBO.")
        return float(np.sum(np.triu(qubo_matrix) * np.outer(bits, bits)))

    def decode_solution(self, bitstring: str) -> UFLPSolution:
        """Decode a bitstring and check assignment and facility-opening constraints."""
        if len(bitstring) != self.total_qubits or not set(bitstring) <= {"0", "1"}:
            raise ValueError("Bitstring length or alphabet does not match the instance.")
        mapping = self.get_qubit_indices()
        facilities_open = [
            int(bitstring[mapping["facility"][(j,)]])
            for j in range(self.instance.n_facilities)
        ]
        assignments = []
        assignment_counts = []
        for i in range(self.instance.n_customers):
            selected = [
                j
                for j in range(self.instance.n_facilities)
                if bitstring[mapping["assignment"][(i, j)]] == "1"
            ]
            assignment_counts.append(len(selected))
            assignments.append(selected[0] if len(selected) == 1 else -1)

        feasible = all(count == 1 for count in assignment_counts)
        if feasible:
            feasible = all(facilities_open[j] == 1 for j in assignments)

        fixed_cost = float(
            sum(
                self.instance.fixed_costs[j]
                for j in range(self.instance.n_facilities)
                if facilities_open[j]
            )
        )
        transport_cost = 0.0
        if all(0 <= j < self.instance.n_facilities for j in assignments):
            transport_cost = float(
                sum(
                    self.instance.transport_costs[i, j]
                    for i, j in enumerate(assignments)
                )
            )
        total_cost = fixed_cost + transport_cost
        return UFLPSolution(
            facilities_open=facilities_open,
            assignments=assignments,
            total_cost=total_cost,
            fixed_cost=fixed_cost,
            transport_cost=transport_cost,
            feasible=feasible,
        )


print("QUBO formulator defined.")

## 3. QAOA circuit and local simulation

A QAOA state begins in the uniform superposition \(|+\rangle^{\otimes n}\). Each layer applies problem evolution and then mixer evolution:

\[
|\psi(\boldsymbol{\gamma},\boldsymbol{\beta})\rangle
= U_M(\beta_p)U_P(\gamma_p)\cdots U_M(\beta_1)U_P(\gamma_1)|+\rangle^{\otimes n},
\]

where \(U_P(\gamma)=e^{-i\gamma H_P}\), \(H_P\) assigns each computational-basis bitstring its QUBO energy, and \(U_M(\beta)=e^{-i\beta H_M}\), with \(H_M=\sum_{q=1}^{n}X_q\). The angles are classical parameters. Measurements estimate the expected QUBO energy, and a classical optimizer searches for angles with lower estimated energy.

The depth \(p\) is the number of alternating layers; it is not the number of qubits. A dense state vector has \(2^n\) complex amplitudes, so both memory and much of the simulation work grow exponentially with \(n\). This notebook implements a direct state-vector simulation only for small cases. The qubit limit below is a transparent local resource guard, not an algorithmic property of QAOA.

In [ ]:
class ClassicalQuantumSimulator:
    """Small-instance state-vector simulator for QAOA circuit operations."""

    MAX_STATEVECTOR_QUBITS = 20

    def __init__(self, n_qubits: int):
        self.n_qubits = n_qubits

    def _check_state_size(self) -> None:
        if self.n_qubits > self.MAX_STATEVECTOR_QUBITS:
            raise MemoryError(
                f"Dense state-vector simulation is limited to "
                f"{self.MAX_STATEVECTOR_QUBITS} qubits in this notebook; "
                f"the requested circuit has {self.n_qubits} qubits."
            )

    def apply_single_qubit_gate(
        self, state: np.ndarray, gate: np.ndarray, target_qubit: int
    ) -> np.ndarray:
        """Apply a gate by reshaping the state instead of forming a 2^n matrix."""
        self._check_state_size()
        if not 0 <= target_qubit < self.n_qubits:
            raise IndexError("Target qubit is outside the simulated register.")
        tensor = state.reshape((2,) * self.n_qubits)
        moved = np.moveaxis(tensor, target_qubit, 0)
        evolved = gate @ moved.reshape(2, -1)
        return np.moveaxis(evolved.reshape(moved.shape), 0, target_qubit).reshape(-1)

    def hadamard_gate(self, state: np.ndarray, target_qubit: int) -> np.ndarray:
        gate = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
        return self.apply_single_qubit_gate(state, gate, target_qubit)

    def rz_gate(self, state: np.ndarray, target_qubit: int, angle: float) -> np.ndarray:
        gate = np.diag([np.exp(-1j * angle / 2), np.exp(1j * angle / 2)])
        return self.apply_single_qubit_gate(state, gate, target_qubit)

    def rx_gate(self, state: np.ndarray, target_qubit: int, angle: float) -> np.ndarray:
        gate = np.array(
            [
                [np.cos(angle / 2), -1j * np.sin(angle / 2)],
                [-1j * np.sin(angle / 2), np.cos(angle / 2)],
            ]
        )
        return self.apply_single_qubit_gate(state, gate, target_qubit)

    def apply_problem_hamiltonian(
        self, state: np.ndarray, qubo_matrix: np.ndarray, gamma: float
    ) -> np.ndarray:
        """Apply the exact diagonal phase exp(-i*gamma*E(z)) to each basis state."""
        self._check_state_size()
        indices = np.arange(2**self.n_qubits, dtype=np.uint64)
        shifts = np.arange(self.n_qubits - 1, -1, -1, dtype=np.uint64)
        bits = ((indices[:, None] >> shifts) & 1).astype(float)
        upper = np.triu(qubo_matrix)
        energies = np.einsum("bi,ij,bj->b", bits, upper, bits, optimize=True)
        return state * np.exp(-1j * gamma * energies)

    def measure_state(self, state: np.ndarray, shots: int) -> Dict[str, int]:
        """Sample computational-basis bitstrings according to state probabilities."""
        self._check_state_size()
        probabilities = np.abs(state) ** 2
        probabilities /= probabilities.sum()
        indices, counts = np.unique(
            np.random.choice(len(state), size=shots, p=probabilities, replace=True),
            return_counts=True,
        )
        return {
            format(int(index), f"0{self.n_qubits}b"): int(count)
            for index, count in zip(indices, counts)
        }


print("Local state-vector simulator defined with an explicit qubit limit.")

In [ ]:
class QAOASolver:
    """QAOA circuit construction and hybrid parameter optimization."""

    def __init__(
        self,
        instance: UFLPInstance,
        params: QAOAParameters,
        hw_config: QuantumHardwareConfig,
    ):
        self.instance = instance
        self.params = params
        self.hw_config = hw_config
        self.qubo_formulator = QUBOFormulator(instance, params.penalty_weight)
        self.qubo_matrix = self.qubo_formulator.formulate_qubo()
        self.simulator = ClassicalQuantumSimulator(self.qubo_formulator.total_qubits)
        self.optimization_history = []
        self.best_solution: Optional[UFLPSolution] = None
        self.best_cost = float("inf")

    def create_initial_state(self) -> np.ndarray:
        """Prepare |+> on every qubit from the all-zero computational state."""
        self.simulator._check_state_size()
        state = np.zeros(2**self.qubo_formulator.total_qubits, dtype=complex)
        state[0] = 1.0
        for qubit in range(self.qubo_formulator.total_qubits):
            state = self.simulator.hadamard_gate(state, qubit)
        return state

    def apply_qaoa_layer(
        self, state: np.ndarray, gamma: float, beta: float
    ) -> np.ndarray:
        state = self.simulator.apply_problem_hamiltonian(
            state, self.qubo_matrix, gamma
        )
        for qubit in range(self.qubo_formulator.total_qubits):
            state = self.simulator.rx_gate(state, qubit, 2 * beta)
        return state

    def _prepare_state(self, params_array: np.ndarray) -> np.ndarray:
        state = self.create_initial_state()
        for layer in range(self.params.circuit_depth):
            state = self.apply_qaoa_layer(
                state, params_array[2 * layer], params_array[2 * layer + 1]
            )
        return state

    def evaluate_qaoa_parameters(self, params_array: np.ndarray) -> float:
        """Estimate expected QUBO energy from the configured number of shots."""
        state = self._prepare_state(params_array)
        measurements = self.simulator.measure_state(state, self.params.shots)
        weighted_energy = sum(
            self.qubo_formulator.qubo_energy(bitstring, self.qubo_matrix) * count
            for bitstring, count in measurements.items()
        )
        expected_energy = weighted_energy / self.params.shots

        for bitstring, count in measurements.items():
            solution = self.qubo_formulator.decode_solution(bitstring)
            if solution.feasible and solution.total_cost < self.best_cost:
                self.best_cost = solution.total_cost
                self.best_solution = solution
        return expected_energy

    def optimize_parameters(self) -> np.ndarray:
        """Use the configured classical optimizer to search QAOA angles."""
        if self.params.optimizer != "COBYLA":
            raise ValueError("This notebook's solver is configured for COBYLA.")
        initial_params = np.random.uniform(
            0, np.pi / 2, 2 * self.params.circuit_depth
        )

        def callback(parameters: np.ndarray) -> None:
            self.optimization_history.append(
                self.evaluate_qaoa_parameters(parameters)
            )

        result = minimize(
            fun=self.evaluate_qaoa_parameters,
            x0=initial_params,
            method="COBYLA",
            options={"maxiter": self.params.max_iterations},
            callback=callback,
        )
        print(f"COBYLA termination message: {result.message}")
        return result.x

    def solve(self) -> Tuple[UFLPSolution, Dict]:
        """Optimize angles, sample the resulting state, and return the best feasible sample."""
        optimal_params = self.optimize_parameters()
        final_state = self._prepare_state(optimal_params)
        final_measurements = self.simulator.measure_state(
            final_state, self.params.shots
        )
        # Include the final samples in the best-feasible-sample selection.
        for bitstring in final_measurements:
            solution = self.qubo_formulator.decode_solution(bitstring)
            if solution.feasible and solution.total_cost < self.best_cost:
                self.best_cost = solution.total_cost
                self.best_solution = solution
        if self.best_solution is None:
            raise RuntimeError("No feasible solution was observed in the QAOA samples.")

        quantum_info = {
            "optimal_params": optimal_params,
            "gammas": optimal_params[::2],
            "betas": optimal_params[1::2],
            "optimization_history": self.optimization_history,
            "measurements": final_measurements,
            "n_qubits": self.qubo_formulator.total_qubits,
            "circuit_depth": self.params.circuit_depth,
            "shots": self.params.shots,
        }
        return self.best_solution, quantum_info


print("QAOA solver defined.")

## 4. Configure the demonstration

The specified teaching parameters are depth \(p=2\), 1,024 measurement shots, COBYLA, 100 maximum optimizer iterations, and penalty weight 1,000. These settings are retained. The instance has 28 binary variables, so the local state-vector limit prevents circuit execution. The configuration and solver remain visible, while the following cell reports the resource limitation without creating a fictitious quantum result.

For a smaller instance within the simulator limit, the same solver methods execute the QAOA circuit, estimate QUBO energy from samples, and retain the lowest-cost feasible sampled UFLP solution. The expected QUBO energy and the original UFLP cost are different quantities because the QUBO includes constraint penalties and omits an irrelevant constant.

In [ ]:
qaoa_params = QAOAParameters(
    circuit_depth=2,
    shots=1024,
    optimizer="COBYLA",
    max_iterations=100,
    penalty_weight=1000.0,
)
hw_config = QuantumHardwareConfig(
    use_aws=False,
    device_arn="",
    s3_bucket="",
    region="us-east-1",
)
qaoa_solver = QAOASolver(quantum_instance, qaoa_params, hw_config)
quantum_solution = None
quantum_info = None

if qaoa_solver.qubo_formulator.total_qubits <= qaoa_solver.simulator.MAX_STATEVECTOR_QUBITS:
    quantum_solution, quantum_info = qaoa_solver.solve()
    print("QAOA completed on the local state-vector simulator.")
    print(f"Best sampled feasible cost: {quantum_solution.total_cost:.1f}")
else:
    print(
        f"Local QAOA simulation not run: {qaoa_solver.qubo_formulator.total_qubits} "
        f"qubits exceed the state-vector limit of "
        f"{qaoa_solver.simulator.MAX_STATEVECTOR_QUBITS}."
    )
    print("No quantum angles, samples, or QAOA solution are reported.")

## 5. Classical comparison for the small instance

The exact benchmark enumerates every nonempty facility-opening subset. For each subset, it assigns each customer to its cheapest open facility; this is optimal for that fixed subset because there are no capacity constraints and transportation costs are additive by customer. The method therefore examines \(2^{|J|}-1\) facility subsets, with \(O(|I||J|)\) assignment work per subset. It is suitable for this small example, not a replacement for the QAOA method being studied.

A simple greedy solution is also included as a heuristic reference. The comparison reports original UFLP costs, not penalized QUBO energies. If QAOA was not executed, its row and comparisons are omitted rather than inferred.

In [ ]:
def compare_quantum_with_classical(
    quantum_instance: UFLPInstance,
    quantum_solution: Optional[UFLPSolution] = None,
) -> Dict[str, UFLPSolution]:
    """Compare available QAOA output with greedy and exact small-instance results."""

    def greedy_classical_solution() -> UFLPSolution:
        facilities_open = [0] * quantum_instance.n_facilities
        assignments = [-1] * quantum_instance.n_customers
        ranked = []
        for j in range(quantum_instance.n_facilities):
            score = quantum_instance.fixed_costs[j] + quantum_instance.n_customers * np.mean(
                quantum_instance.transport_costs[:, j]
            )
            ranked.append((j, score))

        for j, _ in sorted(ranked, key=lambda item: item[1]):
            improved = False
            for i in range(quantum_instance.n_customers):
                current = (
                    quantum_instance.transport_costs[i, assignments[i]]
                    if assignments[i] >= 0
                    else float("inf")
                )
                if quantum_instance.transport_costs[i, j] < current:
                    assignments[i] = j
                    improved = True
            if improved:
                facilities_open[j] = 1

        fixed_cost = float(
            sum(
                quantum_instance.fixed_costs[j]
                for j in range(quantum_instance.n_facilities)
                if facilities_open[j]
            )
        )
        transport_cost = float(
            sum(
                quantum_instance.transport_costs[i, assignments[i]]
                for i in range(quantum_instance.n_customers)
            )
        )
        return UFLPSolution(
            facilities_open,
            assignments,
            fixed_cost + transport_cost,
            fixed_cost,
            transport_cost,
            True,
        )

    def exhaustive_optimal_solution() -> UFLPSolution:
        best_solution = None
        for facility_mask in range(1, 2 ** quantum_instance.n_facilities):
            opened = [
                (facility_mask >> j) & 1
                for j in range(quantum_instance.n_facilities)
            ]
            assignments = []
            for i in range(quantum_instance.n_customers):
                open_indices = [
                    j for j, is_open in enumerate(opened) if is_open
                ]
                assignment = min(
                    open_indices,
                    key=lambda j: quantum_instance.transport_costs[i, j],
                )
                assignments.append(assignment)
            fixed_cost = float(
                sum(
                    quantum_instance.fixed_costs[j]
                    for j, is_open in enumerate(opened)
                    if is_open
                )
            )
            transport_cost = float(
                sum(
                    quantum_instance.transport_costs[i, j]
                    for i, j in enumerate(assignments)
                )
            )
            candidate = UFLPSolution(
                opened,
                assignments,
                fixed_cost + transport_cost,
                fixed_cost,
                transport_cost,
                True,
            )
            if best_solution is None or candidate.total_cost < best_solution.total_cost:
                best_solution = candidate
        assert best_solution is not None
        return best_solution

    solutions = {
        "Classical optimal (exhaustive)": exhaustive_optimal_solution(),
        "Greedy heuristic": greedy_classical_solution(),
    }
    if quantum_solution is not None:
        solutions["QAOA sampled solution"] = quantum_solution

    print("=== UFLP COST COMPARISON ===")
    print(f"{'Method':<32} {'Total':>10} {'Fixed':>10} {'Transport':>12} {'Facilities':>12}")
    for name, solution in solutions.items():
        assert np.isclose(
            solution.total_cost, solution.fixed_cost + solution.transport_cost
        )
        print(
            f"{name:<32} {solution.total_cost:>10.1f} "
            f"{solution.fixed_cost:>10.1f} {solution.transport_cost:>12.1f} "
            f"{sum(solution.facilities_open):>12d}"
        )

    return solutions


classical_comparison = compare_quantum_with_classical(
    quantum_instance, quantum_solution
)

## 6. Interpreting QAOA measurements

A measurement frequency is the number of shots that produced a particular bitstring. Frequency is not a cost, and a frequently measured bitstring is not necessarily feasible. Feasibility requires exactly one assignment bit for each customer and an open facility for every assigned customer. The decoder checks both conditions. The histogram below is generated only when QAOA actually runs; infeasible samples are kept visible rather than silently treated as feasible solutions.

In [ ]:
def visualize_quantum_results(
    quantum_info: Dict,
    quantum_solution: UFLPSolution,
    qubo_formulator: QUBOFormulator,
) -> None:
    """Plot optimizer history, QAOA angles, and sampled feasible costs."""
    measurements = quantum_info["measurements"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    history = quantum_info["optimization_history"]
    if history:
        axes[0].plot(range(1, len(history) + 1), history, marker="o")
    axes[0].set_title("COBYLA objective estimates")
    axes[0].set_xlabel("Callback evaluation")
    axes[0].set_ylabel("Estimated QUBO energy")
    axes[0].grid(True, alpha=0.3)

    layers = np.arange(1, len(quantum_info["gammas"]) + 1)
    axes[1].plot(layers, quantum_info["gammas"], "o-", label="Gamma (problem)")
    axes[1].plot(layers, quantum_info["betas"], "s-", label="Beta (mixer)")
    axes[1].set_title("Optimized QAOA angles")
    axes[1].set_xlabel("Layer")
    axes[1].set_ylabel("Angle (radians)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    feasible_costs = []
    infeasible_counts = 0
    for bitstring, count in measurements.items():
        solution = qubo_formulator.decode_solution(bitstring)
        if solution.feasible:
            feasible_costs.extend([solution.total_cost] * count)
        else:
            infeasible_counts += count

    if feasible_costs:
        axes[2].hist(feasible_costs, bins="auto", alpha=0.75, label="Feasible sampled costs")
        axes[2].axvline(
            quantum_solution.total_cost,
            color="black",
            linestyle="--",
            label="Best feasible sampled cost",
        )
        axes[2].legend()
    else:
        axes[2].text(0.5, 0.5, "No feasible samples", ha="center", va="center")
    axes[2].set_title(f"Sampled UFLP costs ({infeasible_counts} infeasible shots)")
    axes[2].set_xlabel("Original UFLP cost")
    axes[2].set_ylabel("Shot count")
    axes[2].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

    total_shots = sum(measurements.values())
    feasible_shots = sum(
        count
        for bitstring, count in measurements.items()
        if qubo_formulator.decode_solution(bitstring).feasible
    )
    print(f"Total measurement shots: {total_shots}")
    print(f"Feasible measurement shots: {feasible_shots}")
    if total_shots:
        print(f"Feasibility rate: {100 * feasible_shots / total_shots:.1f}%")
    print(f"Best feasible sampled UFLP cost: {quantum_solution.total_cost:.1f}")


if quantum_info is not None and quantum_solution is not None:
    visualize_quantum_results(
        quantum_info, quantum_solution, qaoa_solver.qubo_formulator
    )
else:
    print("QAOA measurement visualizations are unavailable because no circuit was run.")

## 7. Qubit and state-vector scaling

With one binary variable per facility-opening decision and one per customer-facility assignment, this encoding requires

\[
n = |J| + |I||J| = |J|(1+|I|)
\]

qubits. A dense state vector contains \(2^n\) complex amplitudes. This count describes the memory dimension of this particular local simulation representation; it is not a classical runtime benchmark, an estimate of QAOA solution quality, or evidence of quantum speedup. Actual hardware feasibility also depends on connectivity, gate implementation, noise, measurement requirements, and device-specific constraints.

In [ ]:
def analyze_quantum_scalability() -> List[Dict[str, int]]:
    """Report encoding size and dense state-vector dimension for example sizes."""
    problem_sizes = [(4, 6), (6, 10), (8, 15), (10, 20), (15, 25)]
    results = []
    print("=== ENCODING AND STATE-VECTOR SIZE ===")
    print(f"{'Facilities':>10} {'Customers':>10} {'Qubits':>10} {'Amplitudes':>22}")
    for n_facilities, n_customers in problem_sizes:
        n_qubits = n_facilities * (n_customers + 1)
        amplitudes = 2**n_qubits
        results.append(
            {
                "facilities": n_facilities,
                "customers": n_customers,
                "qubits": n_qubits,
                "amplitudes": amplitudes,
            }
        )
        print(
            f"{n_facilities:>10} {n_customers:>10} {n_qubits:>10} "
            f"{amplitudes:>22.3e}"
        )

    facilities = [row["facilities"] for row in results]
    qubits = [row["qubits"] for row in results]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(facilities, qubits, "o-", label="Qubits in this encoding")
    ax.axhline(
        y=ClassicalQuantumSimulator.MAX_STATEVECTOR_QUBITS,
        color="red",
        linestyle="--",
        label="Notebook local simulator limit",
    )
    ax.set_xlabel("Number of facilities")
    ax.set_ylabel("Qubits")
    ax.set_title("Qubit count for the stated example sizes")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return results


scalability_results = analyze_quantum_scalability()

## 8. Summary and practical considerations

This notebook follows the QAOA workflow: formulate the UFLP as a penalized QUBO, encode its energy in a problem Hamiltonian, alternate problem and mixer evolution, and use classical optimization of circuit angles in a hybrid loop. The code also distinguishes the penalized QUBO objective from the original UFLP cost and validates constraints when decoding measurements.

The supplied instance has 28 binary variables. A dense state-vector representation therefore requires \(2^{28}\) complex amplitudes before accounting for temporary arrays and circuit operations. The local simulator does not run this instance, and no QAOA angles, measurement frequencies, or quantum solution are claimed. The exact classical benchmark is included only to interpret this small UFLP instance; its enumeration of facility subsets is not a quantum-versus-classical performance experiment.

### Modeling and implementation limitations
- The model has no facility-capacity constraints, uncertainty, inventory, or time dimension.
- The QUBO penalty weight affects the energy landscape and should be assessed for a particular instance; feasibility is explicitly checked after measurement.
- The simulator is a dense local state-vector implementation with a stated qubit limit. It is educational and is not a substitute for a device-specific circuit implementation.
- The hardware configuration is not connected to Amazon Braket. No credentials, SDK integration, cloud execution, or hardware results are provided.
- QAOA is a heuristic optimization approach in this setting. Superposition and interference do not by themselves imply an exponential speedup, and this notebook makes no quantum-advantage claim.

### Natural extensions
Within the same modeling and QAOA framework, a study could add explicitly defined UFLP constraints, test penalty-weight sensitivity, or run a small instance on a supported simulator or quantum device. Any performance claim would need reproducible instances, resource accounting, and a fair classical benchmark.